In [3]:
import warnings
warnings.filterwarnings("ignore")

In [31]:
# Import required packages

import numpy as np
import pandas as pd
import mlba
import matplotlib.pyplot as plt
import seaborn as sns
import scipy

from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix, confusion_matrix 
from imblearn.over_sampling import SMOTENC
from sklearn.preprocessing import StandardScaler
from sklearn import preprocessing
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

## Modeling (Mia Arnold)

Predicting Credit Card Defaults

Mia Arnold - Katia Paredes - Harshali Gaikwad

Shirley Marcos School of Engineering, University of San Diego

ADS-505 Applied Data Science for Buisness

October 19th, 2026

## Table of Contents
1. [Data Preprocessing](#Data-Preprocessing)
2. [Modeling with Demographic Info](#Modeling-with-Demographic-Info)
3. [Modeling without Demographic Info](#Modeling-without-Demographic-Info)
4. [Model Performance](#Model-Performance)
5. [Model Selection](#Model-Selection)
6. [Discussion](#Discussion)

# Data Preprocessing

In [5]:
# load data

df_1 = pd.read_csv("C:/Users/princ/OneDrive/Desktop/GitHub/ads505-final-project/data/credit_default_engineered.csv")
df_2 = df_1.drop(columns=['SEX','MARRIAGE','AGE'])

print(f'Shape of Data: {df_1.shape}')
print(f'Shape of Data w/o Demographics: {df_2.shape}')

Shape of Data: (30000, 40)
Shape of Data w/o Demographics: (30000, 37)


In [9]:
# test train split

train_1, holdout_1 = train_test_split(df_1, test_size=0.4, stratify=df_1['DEFAULT'], random_state=42)
holdout_1, validate_1 = train_test_split(holdout_1, test_size=0.5, stratify=holdout_1['DEFAULT'], random_state=42)
train_2, holdout_2 = train_test_split(df_2, test_size=0.4, stratify=df_2['DEFAULT'], random_state=42)
holdout_2, validate_2 = train_test_split(holdout_2, test_size=0.5, stratify=holdout_2['DEFAULT'], random_state=42)


# separate predictors and outcome

train_1_X = train_1.drop(columns=['DEFAULT_label','DEFAULT'])
train_1_y = train_1['DEFAULT']
holdout_1_X = holdout_1.drop(columns=['DEFAULT_label','DEFAULT'])
holdout_1_y = holdout_1['DEFAULT']
validate_1_X = validate_1.drop(columns=['DEFAULT_label','DEFAULT'])
validate_1_y = validate_1['DEFAULT']

train_2_X = train_2.drop(columns=['DEFAULT_label','DEFAULT'])
train_2_y = train_2['DEFAULT']
holdout_2_X = holdout_2.drop(columns=['DEFAULT_label','DEFAULT'])
holdout_2_y = holdout_2['DEFAULT']
validate_2_X = validate_2.drop(columns=['DEFAULT_label','DEFAULT'])
validate_2_y = validate_2['DEFAULT']

print(f'Shape of training data w/ Demo: {train_1_X.shape[0]}')
print(f'Shape of testing data w/ Demo: {holdout_1_X.shape[0]}')
print(f'Shape of validating data w/ Demo: {validate_1_X.shape[0]}')

print(f'Shape of training data w/o Demo: {train_2_X.shape[0]}')
print(f'Shape of testing data w/o Demo: {holdout_2_X.shape[0]}')
print(f'Shape of validating data w/o Demo: {validate_2_X.shape[0]}')

Shape of training data w/ Demo: 18000
Shape of testing data w/ Demo: 6000
Shape of validating data w/ Demo: 6000
Shape of training data w/o Demo: 18000
Shape of testing data w/o Demo: 6000
Shape of validating data w/o Demo: 6000


In [10]:
# SMOTE oversampling to address class imbalance

cat_cols = df_1.select_dtypes(include=['object', 'category', 'string']).columns.drop(['DEFAULT_label','Default'], errors='ignore').tolist()

cat_indices_1 = [train_1_X.columns.get_loc(col) for col in cat_cols]
cat_indices_2 = [train_2_X.columns.get_loc(col) for col in cat_cols]

print(f'Data Before Oversampling: {train_1_y.shape[0]}')
print(f'Data w/o Demo Info Before Oversampling: {train_2_y.shape[0]}')

smote_1 = SMOTENC(categorical_features=cat_indices_1, sampling_strategy='minority', random_state=42)

train_1_X, train_1_y = smote_1.fit_resample(train_1_X, train_1_y)

smote_2 = SMOTENC(categorical_features=cat_indices_2, sampling_strategy='minority', random_state=42)

train_2_X, train_2_y = smote_2.fit_resample(train_2_X, train_2_y)

print(f'Data After Oversampling: {train_1_y.shape[0]}')
print(f'Data w/o Demo Info After Oversampling: {train_2_y.shape[0]}')

Data Before Oversampling: 18000
Data w/o Demo Info Before Oversampling: 18000
Data After Oversampling: 28036
Data w/o Demo Info After Oversampling: 28036


In [11]:
# create dummy variables for categorical data

train_1_X_encoded = pd.get_dummies(train_1_X)
holdout_1_X_encoded = pd.get_dummies(holdout_1_X)
holdout_1_X_encoded = holdout_1_X_encoded.reindex(columns=train_1_X_encoded.columns, fill_value=0)
validate_1_X_encoded = pd.get_dummies(validate_1_X)
validate_1_X_encoded = validate_1_X_encoded.reindex(columns=train_1_X_encoded.columns, fill_value=0)

train_2_X_encoded = pd.get_dummies(train_2_X)
holdout_2_X_encoded = pd.get_dummies(holdout_2_X)
holdout_2_X_encoded = holdout_2_X_encoded.reindex(columns=train_2_X_encoded.columns, fill_value=0)
validate_2_X_encoded = pd.get_dummies(validate_2_X)
validate_2_X_encoded = validate_2_X_encoded.reindex(columns=train_2_X_encoded.columns, fill_value=0)

In [13]:
# normalize

scaler = StandardScaler()

train_1_X_norm = scaler.fit_transform(train_1_X_encoded)
holdout_1_X_norm = scaler.transform(holdout_1_X_encoded)
validate_1_X_norm = scaler.transform(validate_1_X_encoded)

train_2_X_norm = scaler.fit_transform(train_2_X_encoded)
holdout_2_X_norm = scaler.transform(holdout_2_X_encoded)
validate_2_X_norm = scaler.transform(validate_2_X_encoded)

# Modeling with Demographic Info

In [33]:
# Baseline model

baseline = DummyClassifier(strategy="constant", constant=0)

baseline.fit(train_1_X, train_1_y)

base_1_pred = baseline.predict(holdout_1_X)

mlba.classificationSummary(y_true=holdout_1_y, y_pred=base_1_pred)

Confusion Matrix (Accuracy 0.7788)

       Prediction
Actual    0    1
     0 4673    0
     1 1327    0


In [14]:
# Adaboost

ada_1 = AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=1), n_estimators=500, learning_rate=0.10, random_state=42)

ada_1.fit(train_1_X_encoded, train_1_y)

ada_1_pred = ada_1.predict(holdout_1_X_encoded)

mlba.classificationSummary(y_true=holdout_1_y, y_pred=ada_1_pred)

Confusion Matrix (Accuracy 0.7520)

       Prediction
Actual    0    1
     0 3781  892
     1  596  731


In [15]:
# Linear Discriminant Analysis (LDA)

da_1 = LinearDiscriminantAnalysis()

da_1.fit(train_1_X_encoded, train_1_y)

da_1_pred = da_1.predict(holdout_1_X_encoded)

mlba.classificationSummary(y_true=holdout_1_y, y_pred=da_1_pred)

Confusion Matrix (Accuracy 0.7493)

       Prediction
Actual    0    1
     0 3850  823
     1  681  646


In [30]:
# Random Forest


In [ ]:
# Logistic Regression


In [ ]:
# Neural Network


In [ ]:
# Decision Tree


# Modeling without Demographic Info

In [34]:
# Baseline model

baseline = DummyClassifier(strategy="constant", constant=0)

baseline.fit(train_2_X, train_2_y)

base_2_pred = baseline.predict(holdout_2_X)

mlba.classificationSummary(y_true=holdout_2_y, y_pred=base_2_pred)

Confusion Matrix (Accuracy 0.7788)

       Prediction
Actual    0    1
     0 4673    0
     1 1327    0


In [16]:
# Adaboost

ada_2 = AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=1), n_estimators=500, learning_rate=0.10, random_state=42)

ada_2.fit(train_2_X_encoded, train_2_y)

ada_2_pred = ada_2.predict(holdout_2_X_encoded)

mlba.classificationSummary(y_true=holdout_2_y, y_pred=ada_2_pred)

Confusion Matrix (Accuracy 0.7537)

       Prediction
Actual    0    1
     0 3767  906
     1  572  755


In [17]:
# Linear Discriminant Analysis (LDA)

da_2 = LinearDiscriminantAnalysis()

da_2.fit(train_2_X_encoded, train_2_y)

da_2_pred = da_2.predict(holdout_2_X_encoded)

mlba.classificationSummary(y_true=holdout_2_y, y_pred=da_2_pred)

Confusion Matrix (Accuracy 0.7028)

       Prediction
Actual    0    1
     0 3440 1233
     1  550  777


In [30]:
# Random Forest


In [ ]:
# Logistic Regression


In [ ]:
# Neural Network


In [ ]:
# Decision Tree


# Model Performance

In [35]:
# Baseline w/ Demo

acc = accuracy_score(holdout_1_y, base_1_pred)
prec = precision_score(holdout_1_y, base_1_pred)
rec = recall_score(holdout_1_y, base_1_pred)

tn, fp, fn, tp = confusion_matrix(holdout_1_y, base_1_pred).ravel()

fpr = fp / (fp + tn)

F1 = 2*tp/(2*tp + fp + fn)

eval_metrics = [["Accuracy", acc],["Precision", prec],["Recall", rec],["FPR", fpr],["F1 Score", F1]]
eval_metrics_ada_1=pd.DataFrame(eval_metrics, columns=["Metric", "Value"])
eval_metrics_ada_1

,Metric,Value
0,Accuracy,0.778833
1,Precision,0.000000
2,Recall,0.000000
3,FPR,0.000000
4,F1 Score,0.000000


In [27]:
# Adaboost w/ Demo

acc = accuracy_score(holdout_1_y, ada_1_pred)
prec = precision_score(holdout_1_y, ada_1_pred)
rec = recall_score(holdout_1_y, ada_1_pred)

tn, fp, fn, tp = confusion_matrix(holdout_1_y, ada_1_pred).ravel()

fpr = fp / (fp + tn)

F1 = 2*tp/(2*tp + fp + fn)

eval_metrics = [["Accuracy", acc],["Precision", prec],["Recall", rec],["FPR", fpr],["F1 Score", F1]]
eval_metrics_ada_1=pd.DataFrame(eval_metrics, columns=["Metric", "Value"])
eval_metrics_ada_1

,Metric,Value
0,Accuracy,0.752000
1,Precision,0.450400
2,Recall,0.550867
3,FPR,0.190884
4,F1 Score,0.495593


In [28]:
# Linear Discriminant Analysis (LDA) w/ Demo

acc = accuracy_score(holdout_1_y, da_1_pred)
prec = precision_score(holdout_1_y, da_1_pred)
rec = recall_score(holdout_1_y, da_1_pred)

tn, fp, fn, tp = confusion_matrix(holdout_1_y, da_1_pred).ravel()

fpr = fp / (fp + tn)

F1 = 2*tp/(2*tp + fp + fn)

eval_metrics = [["Accuracy", acc],["Precision", prec],["Recall", rec],["FPR", fpr],["F1 Score", F1]]
eval_metrics_da_1=pd.DataFrame(eval_metrics, columns=["Metric", "Value"])
eval_metrics_da_1

,Metric,Value
0,Accuracy,0.749333
1,Precision,0.439755
2,Recall,0.486812
3,FPR,0.176118
4,F1 Score,0.462089


In [36]:
# Baseline w/o Demo

acc = accuracy_score(holdout_2_y, base_2_pred)
prec = precision_score(holdout_2_y, base_2_pred)
rec = recall_score(holdout_2_y, base_2_pred)

tn, fp, fn, tp = confusion_matrix(holdout_2_y, base_2_pred).ravel()

fpr = fp / (fp + tn)

F1 = 2*tp/(2*tp + fp + fn)

eval_metrics = [["Accuracy", acc],["Precision", prec],["Recall", rec],["FPR", fpr],["F1 Score", F1]]
eval_metrics_ada_1=pd.DataFrame(eval_metrics, columns=["Metric", "Value"])
eval_metrics_ada_1

,Metric,Value
0,Accuracy,0.778833
1,Precision,0.000000
2,Recall,0.000000
3,FPR,0.000000
4,F1 Score,0.000000


In [29]:
# Adaboost w/o Demo

acc = accuracy_score(holdout_2_y, ada_2_pred)
prec = precision_score(holdout_2_y, ada_2_pred)
rec = recall_score(holdout_2_y, ada_2_pred)

tn, fp, fn, tp = confusion_matrix(holdout_2_y, ada_2_pred).ravel()

fpr = fp / (fp + tn)

F1 = 2*tp/(2*tp + fp + fn)

eval_metrics = [["Accuracy", acc],["Precision", prec],["Recall", rec],["FPR", fpr],["F1 Score", F1]]
eval_metrics_ada_2=pd.DataFrame(eval_metrics, columns=["Metric", "Value"])
eval_metrics_ada_2

,Metric,Value
0,Accuracy,0.753667
1,Precision,0.454545
2,Recall,0.568953
3,FPR,0.193880
4,F1 Score,0.505355


In [30]:
# Linear Discriminant Analysis (LDA) w/o Demo

acc = accuracy_score(holdout_2_y, da_2_pred)
prec = precision_score(holdout_2_y, da_2_pred)
rec = recall_score(holdout_2_y, da_2_pred)

tn, fp, fn, tp = confusion_matrix(holdout_2_y, da_2_pred).ravel()

fpr = fp / (fp + tn)

F1 = 2*tp/(2*tp + fp + fn)

eval_metrics = [["Accuracy", acc],["Precision", prec],["Recall", rec],["FPR", fpr],["F1 Score", F1]]
eval_metrics_da_2=pd.DataFrame(eval_metrics, columns=["Metric", "Value"])
eval_metrics_da_2

,Metric,Value
0,Accuracy,0.702833
1,Precision,0.386567
2,Recall,0.585531
3,FPR,0.263856
4,F1 Score,0.465688


In [30]:
# Random Forest


In [ ]:
# Logistic Regression


In [ ]:
# Neural Network


In [ ]:
# Decision Tree


# Model Selection

The linear discriminant analysis (LDA) and the ada boost models performed similarly for both dataset. Neither model\
out preformed the baseline in terms of accuracy. Yet, for this problem accuracy isn't a strong metric. This is\
because of class imbalance. Additionally, the buisness goal is to detect the positive class correctly and the\
baseline model fails to predict the any of the positive class values. That being said, Adaboost had higher precision\
and recall than LDA. However, it also had a higher false positive rate. The cost of a false positve is the\
additional resources being spent to discourage default payments on these individuals. The F1 score which balances\
recall and false positive rate is slightly higher for the Adaboost model.However, an F1 score below 0.6 is often \
considered low and both models are victum to this. Therefore, parameter tuning is suggested to improve the models\
further.

# Discussion

Neither model beat the baseline model in terms of accuracy. When comparing the F1 scores, Adaboost did\
slightly better than LDA. However both scores were poor. Until parameter tuning is performed, neither\
model is reccomended. If their preformance is improved, they can be compared to each other as well as\
other models including Random Forest, Neural Networks, Logistic Regression, and Descion Trees.\
Additionally, cumlative gains charts should be employed to better compare the models and determine\
reccomendations for deployment.